# 🏠 Estato AI: PyTorch LSTM Model Training (Returns-Based & Multi-Region)

Welcome to the production-grade training playground! This notebook implements two critical improvements to prevent model lag and overfitting:
1. **Returns-Based Forecasting**: Instead of predicting raw prices (which causes the LSTM to fall into a lazy lag/echo shortcut where `price[t] ≈ price[t-1]`), we train the model to predict the **monthly percentage return** (rate of change).
2. **Multi-Region Data Aggregation**: To prevent overfitting to a single city's noise, we extract and train on the top 30 US Metropolitan markets simultaneously, multiplying our training dataset size by 30x.
3. **Zero Data Leakage**: We fit the scalers strictly on training sequences to avoid test-set data leakage.
4. **NaN Safety (FIXED)**: We clean all missing (NaN) cells from historical rows to prevent gradient explosion and NaN losses.

In [ ]:
# 1. Setup & Imports
import os
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt

print("PyTorch Version:", torch.__version__)
print("CUDA / GPU Available:", torch.cuda.is_available())

In [ ]:
# 2. Load Zillow Dataset
csv_path = "data/zhvi_metro.csv"

if not os.path.exists(csv_path):
    print("Dataset not found! Let's download it.")
    from src.download_data import download_zillow_data
    download_zillow_data()

df = pd.read_csv(csv_path)
print(f"Loaded Zillow Dataset. Shape: {df.shape}")

## 3. Data Processing: Returns Calculation & Multi-Region Windows
We extract the top 30 Metro areas, calculate their monthly return sequences, scale them independently, and concatenate them into a unified dataset.

In [ ]:
def create_sequences(data, seq_length=12):
    xs = []
    ys = []
    for i in range(len(data) - seq_length):
        xs.append(data[i:(i + seq_length)])
        ys.append(data[i + seq_length])
    return np.array(xs), np.array(ys)

# Extract only date columns
date_cols = [c for c in df.columns if "-" in c]

# Select top 30 metro regions by SizeRank to avoid small-town market noise
df_top = df.sort_values(by="SizeRank").head(30)

X_train_list, y_train_list = [], []
X_test_list, y_test_list = [], []

seq_length = 12

print("Processing time-series returns for top 30 cities...")
for idx, row in df_top.iterrows():
    raw_prices = row[date_cols].values.astype(float)
    
    # CRITICAL FIX: Clean missing (NaN) cells from historical values
    prices = raw_prices[~np.isnan(raw_prices)]
    
    # Skip regions with insufficient history
    if len(prices) < 24:
        continue
        
    # Calculate monthly percentage returns: (p[t] - p[t-1]) / p[t-1]
    returns = np.diff(prices) / prices[:-1]
    returns = np.nan_to_num(returns, nan=0.0, posinf=0.0, neginf=0.0)
    
    # Split raw returns first to prevent data leakage
    raw_train_size = int(len(returns) * 0.8)
    train_returns = returns[:raw_train_size]
    test_returns = returns[raw_train_size - seq_length:]
    
    # Scale returns between -1 and 1
    scaler = MinMaxScaler(feature_range=(-1, 1))
    scaler.fit(train_returns.reshape(-1, 1))
    
    scaled_train = scaler.transform(train_returns.reshape(-1, 1)).flatten()
    scaled_test = scaler.transform(test_returns.reshape(-1, 1)).flatten()
    
    # Create sequences
    x_tr, y_tr = create_sequences(scaled_train, seq_length)
    x_te, y_te = create_sequences(scaled_test, seq_length)
    
    X_train_list.append(x_tr)
    y_train_list.append(y_tr)
    X_test_list.append(x_te)
    y_test_list.append(y_te)

# Concatenate all regions
X_train_np = np.concatenate(X_train_list, axis=0).reshape(-1, seq_length, 1)
y_train_np = np.concatenate(y_train_list, axis=0).reshape(-1, 1)
X_test_np = np.concatenate(X_test_list, axis=0).reshape(-1, seq_length, 1)
y_test_np = np.concatenate(y_test_list, axis=0).reshape(-1, 1)

# Convert to tensors
X_train, y_train = torch.tensor(X_train_np, dtype=torch.float32), torch.tensor(y_train_np, dtype=torch.float32)
X_test, y_test = torch.tensor(X_test_np, dtype=torch.float32), torch.tensor(y_test_np, dtype=torch.float32)

print(f"Total Training Windows: {X_train.shape[0]} | Testing Windows: {X_test.shape[0]}")
print(f"Sequence shape: {X_train.shape}")

## 4. Define PyTorch LSTM Architecture

In [ ]:
class HousingLSTM(nn.Module):
    def __init__(self, input_size=1, hidden_size=64, num_layers=2, output_size=1, dropout=0.2):
        super(HousingLSTM, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.lstm = nn.LSTM(
            input_size, hidden_size, num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0
        )
        self.fc = nn.Linear(hidden_size, output_size)
        
    def forward(self, x):
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        out, _ = self.lstm(x, (h0, c0))
        out = self.fc(out[:, -1, :])
        return out

model = HousingLSTM()
print(model)

## 5. Model Training Loop ( Adam + MSE )

In [ ]:
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.003)

epochs = 120
train_losses = []
val_losses = []

print("🚀 Training LSTM on returns sequence space...")
for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    
    outputs = model(X_train)
    loss = criterion(outputs, y_train)
    
    loss.backward()
    optimizer.step()
    train_losses.append(loss.item())
    
    # Evaluate on validation
    model.eval()
    with torch.no_grad():
        val_outputs = model(X_test)
        val_loss = criterion(val_outputs, y_test)
        val_losses.append(val_loss.item())
        
    if (epoch + 1) % 15 == 0:
        print(f"Epoch [{epoch+1}/{epochs}] | Train MSE: {loss.item():.6f} | Val MSE: {val_loss.item():.6f}")

print("✅ Training Complete!")

In [ ]:
# Plot the loss convergence curves
plt.figure(figsize=(10, 5))
plt.plot(train_losses, label="Training Loss (Returns)", color="purple")
plt.plot(val_losses, label="Validation Loss (Returns)", color="orange")
plt.title("LSTM Loss Convergence on Percentage Returns")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
plt.show()

## 6. Testing & Price Reconstruction
Now we run inference on the test returns, inverse-transform them back to actual return rates, and use them to reconstruct actual house prices starting from the last known training price. This proves the model is actually predicting trends instead of copying lagged numbers!

In [ ]:
# Let's run evaluation on San Francisco's test segment
sf_data = df[df["RegionName"].str.contains("San Francisco", case=False)]
raw_sf_prices = sf_data[date_cols].values.flatten().astype(float)

# Clean NaNs
sf_prices = raw_sf_prices[~np.isnan(raw_sf_prices)]

sf_returns = np.diff(sf_prices) / sf_prices[:-1]
sf_returns = np.nan_to_num(sf_returns, nan=0.0, posinf=0.0, neginf=0.0)

raw_train_size = int(len(sf_returns) * 0.8)
train_ret = sf_returns[:raw_train_size]
test_ret = sf_returns[raw_train_size - seq_length:]

scaler = MinMaxScaler(feature_range=(-1, 1))
scaler.fit(train_ret.reshape(-1, 1))
scaled_test_ret = scaler.transform(test_ret.reshape(-1, 1)).flatten()

x_te, y_te = create_sequences(scaled_test_ret, seq_length)
x_te_tensor = torch.tensor(x_te.reshape(-1, seq_length, 1), dtype=torch.float32)

model.eval()
with torch.no_grad():
    pred_ret_scaled = model(x_te_tensor).numpy()
    
# Inverse transform to get actual returns
predicted_returns = scaler.inverse_transform(pred_ret_scaled).flatten()
actual_returns = y_te

# Reconstruct actual prices starting from the boundary price
boundary_price = sf_prices[raw_train_size]

reconstructed_actual = []
reconstructed_pred = []

curr_actual = boundary_price
curr_pred = boundary_price

for act_ret, prd_ret in zip(actual_returns, predicted_returns):
    curr_actual = curr_actual * (1 + act_ret)
    curr_pred = curr_pred * (1 + prd_ret)
    reconstructed_actual.append(curr_actual)
    reconstructed_pred.append(curr_pred)

# Plot reconstructed actual prices vs predicted prices
plt.figure(figsize=(12, 6))
plt.plot(reconstructed_actual, label="Actual Prices (SF)", color="green", marker="o")
plt.plot(reconstructed_pred, label="LSTM Predicted Prices (SF)", color="red", linestyle="--", marker="x")
plt.title("Actual vs. Returns-Based Predicted Prices (No Lagging Echo Effect!)")
plt.ylabel("Price ($)")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Export the trained return weights locally
model_dir = "models"
os.makedirs(model_dir, exist_ok=True)
model_save_path = os.path.join(model_dir, "housing_lstm.pth")
torch.save(model.state_dict(), model_save_path)
print(f"🎉 Successfully saved returns-based PyTorch weights to: {model_save_path}")